# Train and Save Model — Black Friday High Spender Classifier

This notebook trains the classification models, compares them, and saves the best one as `model.joblib`.
The saved pipeline is then loaded by `app.py` for inference — **no training happens in `app.py`**.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

SCRIPT_DIR = Path().resolve()
print('Working directory:', SCRIPT_DIR)

## 1. Load Sample Dataset

In [ ]:
df = pd.read_csv(SCRIPT_DIR / 'classification_sample.csv')
print('Shape:', df.shape)
print(df.head())

## 2. Preprocessing

In [ ]:
# Create target variable
median_purchase = df['purchase_amount'].median()
df['high_spender'] = (df['purchase_amount'] > median_purchase).astype(int)

# Drop columns not needed
cols_to_drop = ['transaction_id', 'customer_id', 'product_id',
                'purchase_date', 'purchase_amount', 'original_price', 'purchase_hour']
df_clean = df.drop(columns=cols_to_drop)

# Label Encoding
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
le = LabelEncoder()
for col in categorical_cols:
    df_clean[col] = le.fit_transform(df_clean[col])

# Features and target
FEATURES = [c for c in df_clean.columns if c != 'high_spender']
TARGET   = 'high_spender'

X = df_clean[FEATURES]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Features: {FEATURES}')
print(f'Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows')

## 3. Train Candidate Models

In [ ]:
candidates = {
    'Logistic Regression'    : Pipeline([('clf', LogisticRegression(random_state=42, max_iter=1000))]),
    'Decision Tree (depth=3)': Pipeline([('clf', DecisionTreeClassifier(max_depth=3, random_state=42))]),
    'Decision Tree (depth=10)': Pipeline([('clf', DecisionTreeClassifier(max_depth=10, random_state=42))]),
}

results = []
fitted_pipelines = {}

for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append({'Algorithm': name, 'Accuracy': round(acc, 4)})
    fitted_pipelines[name] = pipe
    print(f'{name}: {acc:.4f}')

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print('\n', results_df)

## 4. Select Winner & Save Artifacts

Decision Tree (depth=10) has highest accuracy but shows signs of overfitting (99.99%).
**Decision Tree (depth=3)** is selected as the best model — excellent accuracy (98.48%) without overfitting.

In [ ]:
WINNER = 'Decision Tree (depth=3)'

# Save model.joblib with all pipelines + metadata
joblib.dump({
    'pipelines'    : fitted_pipelines,
    'winner'       : WINNER,
    'feature_names': FEATURES,
    'target_name'  : TARGET,
    'positive_label': 'High Spender',
    'negative_label': 'Not High Spender',
    'threshold'    : float(median_purchase),
}, SCRIPT_DIR / 'model.joblib')

print(f'model.joblib saved — winner: {WINNER}')

# Save model_comparison.csv for the Model Card tab in app.py
results_df['Winner'] = results_df['Algorithm'] == WINNER
results_df.to_csv(SCRIPT_DIR / 'model_comparison.csv', index=False)
print('model_comparison.csv saved')
print(results_df)